In [2]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS

In [3]:
load_dotenv()

True

In [4]:
PDF_PATH = "data.pdf"  # Replace with your PDF file path
VECTOR_DB_PATH = "vectorstore"

In [5]:
def ingest_docs():

    # 1. Load PDF
    if not os.path.exists(PDF_PATH):
        print(f"❌ Error: File {PDF_PATH} not found.")
        return
    print("Loading Pdf...")
    loader = PyPDFLoader(PDF_PATH)
    docs = loader.load()


    # 2. Split into Chunks
    print("✂️  Chunking documents...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=100,
    )
    splits = text_splitter.split_documents(docs)
    print(f"Created {len(splits)} chunks from document.")



    # 3. Creating Embeddings and Vector Store
    print("💾 Creating vector store...")
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

    vectorstore = FAISS.from_documents(splits, embeddings)
    vectorstore.save_local(VECTOR_DB_PATH)
    print(f"✅ Success! Vector store saved to '{VECTOR_DB_PATH}'")

   
ingest_docs()
    

    

Loading Pdf...
✂️  Chunking documents...
Created 65 chunks from document.
💾 Creating vector store...
✅ Success! Vector store saved to 'vectorstore'


In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def main():

    llm = ChatGoogleGenerativeAI(
        model = "gemini-2.5-flash-lite",
        temperature=0
    )


    vectorstore = FAISS.load_local(
        VECTOR_DB_PATH,
        GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"),
        allow_dangerous_deserialization=True
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={"k":5}
    )

    # 4. Define System Prompt (The "Brain")    
    system_prompt = (
        "You are an assistant for question-answering tasks. "
        "Use the following pieces of retrieved context to answer "
        "the question. If you don't know the answer, answer smoothly as per your preference"
        "answer concise."
        "Use emojis to answer"
        "\n\n"
        """Context:
            {context}

            Question:
            {question}"""
    )

    prompt = ChatPromptTemplate.from_template(system_prompt)

    def format_docs(docs):
        return "\n\n".join([d.page_content for d in docs])

    # 5. Build the Chain using Pipes (|) 
    rag_chain = (
        # Step 1: Fetch data in parallel
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
    
        # Step 2: Pass to Prompt
        | prompt
    
        # Step 3: Pass to LLM
        | llm
    
        # Step 4: Parse output to string
        | StrOutputParser()
    )

    # 6. Run It
    while True:
        query = input("You: ")
        print(f"User: {query}")
        if query.lower() == "exit":
            print("Bot: Good bye! See you again.🙋‍♂️" )
            break
    
        response = rag_chain.invoke(query)
        print(f"Bot: {response}")

if __name__ == "__main__":
    main()



User: hii
Bot: 👋 How can I help you today?
User: tell me something about ai
Bot: 🤖 AI, or Artificial Intelligence, is a rapidly evolving field with a "terminology maze" that can make it confusing to understand the different types. 🤯

Here's a quick breakdown:

*   **Traditional AI:** Basic automation and rule-based systems. Think of it as following a strict recipe. 📜
*   **Non-agentic AI:** These systems assist in tasks but lack autonomy. They don't make independent decisions. 🧑‍🏫
*   **Agentic AI:** This is where it gets interesting! Agentic AI systems are capable of autonomous decision-making and can adapt to new situations. They can proactively problem-solve and deal with ambiguity. 🧠✨
*   **Generative AI:** This type of AI creates new content, like text, images, or music. 🎨🎶

Agentic AI is particularly powerful because it can learn, adapt, and make decisions without constant human input. This leads to more efficient and effective operations in areas like logistics, retail, media, a